In [1]:
import os
os.chdir("../")
%pwd

'C:\\Users\\Sayantan Nandi\\Desktop\\mlops\\developer-burnout-project'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    metric_file_name: Path

In [3]:
from src.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.utils.common import read_yaml, create_directories

In [4]:
class ConfigurationManager:
    def __init__(self,
                 config_path=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                ):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        return ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            model_path=Path(config.model_path),
            metric_file_name=Path(config.metric_file_name)
        )

In [11]:
import pandas as pd
import joblib
import json

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from src.logging import logger
from src.utils.common import create_directories

from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
class ModelEvaluation:
    def __init__(self, config):
        self.config = config
        create_directories([self.config.root_dir])

    def evaluate(self):
        logger.info("Starting model evaluation...")

        # load model
        logger.info("Loading trained model...")
        model = joblib.load(self.config.model_path)

        # test data
        logger.info("Loading test data...")
        test_df = pd.read_csv(f"{self.config.data_path}/test.csv")

        X_test = test_df.drop("burnout_level", axis=1)
        y_test = test_df["burnout_level"]

        logger.info("Data loaded successfully.")

        # predictions
        logger.info("Generating predictions...")
        y_pred = model.predict(X_test)

        # metrics
        logger.info("Calculating evaluation metrics...")

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average="weighted")
        recall = recall_score(y_test, y_pred, average="weighted")
        f1 = f1_score(y_test, y_pred, average="weighted")

        metrics = {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1
        }

        # confusion matrix
        logger.info("Generating confusion matrix...")

        cm = confusion_matrix(y_test, y_pred)

        le = joblib.load(f"{self.config.data_path}/preprocessor/label_encoder.pkl")
        labels = le.classes_

        plt.figure(figsize=(6, 5))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=labels,
            yticklabels=labels
        )

        plt.title("Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")

        cm_path = f"{self.config.root_dir}/confusion_matrix.png"
        plt.savefig(cm_path)
        plt.close()

        logger.info(f"Confusion matrix saved at {cm_path}")

        logger.info(f"Metrics: {metrics}")

        # save metrics
        logger.info("Saving metrics...")

        metrics_df = pd.DataFrame([metrics])
        metrics_df.to_csv(self.config.metric_file_name, index=False)

        logger.info(f"Metrics saved at {self.config.metric_file_name}")

        return metrics

In [15]:
configurationManager = ConfigurationManager()

eval_config = configurationManager.get_model_evaluation_config()
model_eval = ModelEvaluation(eval_config)

model_eval.evaluate()

[2026-04-16 22:51:54,309: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-16 22:51:54,314: INFO: common: yaml file: config\params.yaml loaded successfully]
[2026-04-16 22:51:54,315: INFO: common: created directory at: artifacts]
[2026-04-16 22:51:54,317: INFO: common: created directory at: artifacts/model_evaluation]
[2026-04-16 22:51:54,318: INFO: common: created directory at: artifacts\model_evaluation]
[2026-04-16 22:51:54,318: INFO: 3604590621: Starting model evaluation...]
[2026-04-16 22:51:54,319: INFO: 3604590621: Loading trained model...]
[2026-04-16 22:51:54,343: INFO: 3604590621: Loading test data...]
[2026-04-16 22:51:54,349: INFO: 3604590621: Data loaded successfully.]
[2026-04-16 22:51:54,351: INFO: 3604590621: Generating predictions...]
[2026-04-16 22:51:54,372: INFO: 3604590621: Calculating evaluation metrics...]
[2026-04-16 22:51:54,383: INFO: 3604590621: Generating confusion matrix...]
[2026-04-16 22:51:54,522: INFO: 3604590621: Confusion matr

{'accuracy': 0.9919825072886297,
 'precision': 0.9921070728533544,
 'recall': 0.9919825072886297,
 'f1_score': 0.9919802471910348}